In [7]:
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader
DATASET = 'sviridov/nbadeepfm'


dataset = load_dataset(DATASET)

combined_dataset = concatenate_datasets(list(dataset.values()))
combined_dataset = combined_dataset.with_format('torch')


In [8]:
import polars as pl
import torch


players_df = pl.read_csv('csv/player.csv')

all_ids = players_df['id'].unique()
null_player_id = 0
nba_id_to_idx_mapping = {}
for idx, nba_id in enumerate(all_ids):
    nba_id_to_idx_mapping[nba_id] = idx + 1 # reserve 0 for null player id

nba_id_to_idx_mapping[null_player_id] = 0

max_nba_id = max(nba_id_to_idx_mapping.keys())
lookup_tensor = torch.zeros(max_nba_id + 1, dtype=torch.long)
for nba_id, sequential_idx in nba_id_to_idx_mapping.items():
    lookup_tensor[nba_id] = sequential_idx
    

In [ ]:
import numpy as np

# use external variable unfortunately
def preprocess_function(dataset: DataLoader):
    
    # 1. Combine players into a single lineup token sequence [Batch, 10]
    lineups = torch.stack([
        lookup_tensor[dataset['offensive_players']],
        lookup_tensor[dataset['defensive_players']]
    ], dim=1).view(-1, 10)


    def to_tensor_clean(x, dtype=torch.float32):
        if isinstance(x, list):
            x = [0 if v is None else v for v in x]
        t = torch.tensor(x, dtype=dtype)
        return t.nan_to_num(0)

    # 2. Extract Role IDs (Shooter, Assister, Rebounder, Defender)
    # Ensure these are handled as a single [Batch, 4] tensor

    shooting_player = torch.tensor(dataset['shooting_player']).nan_to_num(0).to(torch.long)
    assisting_player = torch.tensor(dataset['assisting_player']).nan_to_num(0).to(torch.long)
    defending_player = torch.tensor(dataset['defending_player']).nan_to_num(0).to(torch.long)
    rebounding_player = torch.tensor(dataset['rebounding_player']).nan_to_num(0).to(torch.long)


    roles = torch.stack([
        lookup_tensor[shooting_player],
        lookup_tensor[assisting_player],
        lookup_tensor[rebounding_player],
        lookup_tensor[defending_player]
    ], dim=1)

    # 3. Pack Privileged Info (PI) into a single [Batch, 6] float tensor
    pi_stats = torch.stack([
        dataset['shot_distance'].nan_to_num(0.0),
        dataset['is_putback'].to(torch.float32),
        dataset['is_and1'].to(torch.float32),
        dataset['is_freethrow'].to(torch.float32),
        dataset['is_turnover'].to(torch.float32),
        dataset['is_steal'].to(torch.float32)
    ], dim=1)

    return {
        "lineup_ids": lineups,
        "role_ids": roles,
        "pi_stats": pi_stats,
        "labels": dataset['shot_points_scored'].nan_to_num(0).to(torch.long)
    }


# Apply mapping (Fast and memory-efficient)
processed_dataset = combined_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=combined_dataset.column_names
)

# processed_dataset.save_to_disk('nba_processed_dataset.arrow')
processed_dataset.push_to_hub("sviridov/nba-posession-processed")


Map:   0%|          | 0/5347569 [00:00<?, ? examples/s]


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.